## Task 3 - Data Preprocessing and Feature Engineering
This section handles data loading, target discretisation (quantile-based), and feature engineering (ordinal and one-hot encoding).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    classification_report,
)
from statsmodels.stats.contingency_tables import mcnemar

# Point Python at the scikit-eLCS GitHub master folder
# Replaced __file__ with '' for Jupyter compatibility
ELCS_REPO = os.path.join(os.path.dirname(os.path.abspath('')), "scikit-eLCS-master")
if os.path.isdir(ELCS_REPO):
    sys.path.insert(0, ELCS_REPO)
from skeLCS import eLCS

In [ ]:
RAW_DATA_PATH = "../ai_company_adoption.csv"
df = pd.read_csv(RAW_DATA_PATH)
print("Raw dataset shape:", df.shape)

# Quantile-based cutoffs for the target variable (Low, Medium, High)
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)

print("Class balances:")
print(df["productivity_class"].value_counts())

Raw dataset shape: (150000, 43)
Class balances:
productivity_class
1    50031
0    50023
2    49946
Name: count, dtype: int64


In [ ]:
# Build the Preprocessed + Feature-Engineered Matrix
SELECTED_FEATURE_COLUMNS = [
    "ai_adoption_rate", "ai_maturity_score", "ai_failure_rate",
    "ai_training_hours", "task_automation_rate", "ai_adoption_stage", "industry",
]

X_df = df[SELECTED_FEATURE_COLUMNS].copy()

# Ordinal encoding for adoption stage
STAGE_ORDER = {"none": 0, "pilot": 1, "partial": 2, "full": 3}
X_df["ai_adoption_stage"] = X_df["ai_adoption_stage"].map(STAGE_ORDER)

# One-hot encoding for industry
X_df = pd.get_dummies(X_df, columns=["industry"], prefix="industry", dtype=float)

FEATURE_NAMES = list(X_df.columns)
X = X_df.values.astype(float)
y = df["productivity_class"].values.astype(int)

print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (150000, 15) | y shape: (150000,)


## Task 4 - Original and Improved LCS Models (10-Seed Loop)
Running the 10-seed repeated evaluation for both the original eLCS baseline and the improved eLCS system.

In [ ]:
# Repeated-Evaluation Settings
N_SEEDS = 10
TRAIN_SUBSAMPLE_SIZE = 1000
TEST_EVAL_SIZE = 5000
POPULATION_SIZE_N = round(1.5 * TRAIN_SUBSAMPLE_SIZE)
LEARNING_ITERATIONS = 5 * TRAIN_SUBSAMPLE_SIZE

print(f"Population size N = {POPULATION_SIZE_N}, learning_iterations = {LEARNING_ITERATIONS}")

Population size N = 1500, learning_iterations = 5000


In [ ]:
print("--- Training Original eLCS (Preprocessed) ---")
orig_per_seed_acc = []
orig_all_true = []
orig_all_pred = []
orig_last_model = None

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    rng = np.random.RandomState(seed)
    train_subsample_idx = rng.choice(len(X_train), TRAIN_SUBSAMPLE_SIZE, replace=False)
    
    X_train_sub = X_train[train_subsample_idx]
    y_train_sub = y_train[train_subsample_idx]
    X_test_eval = X_test[:TEST_EVAL_SIZE]
    y_test_eval = y_test[:TEST_EVAL_SIZE]

    model = eLCS(
        learning_iterations=LEARNING_ITERATIONS,
        N=POPULATION_SIZE_N,
        random_state=seed,
    )
    model.fit(X_train_sub, y_train_sub)

    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    
    orig_per_seed_acc.append(acc)
    orig_all_true.extend(y_test_eval.tolist())
    orig_all_pred.extend(y_pred.tolist())
    orig_last_model = model

    print(f"Seed {seed}: accuracy = {acc:.4f}")

--- Training Original eLCS (Preprocessed) ---
Seed 0: accuracy = 0.5924
Seed 1: accuracy = 0.5652
Seed 2: accuracy = 0.5664
Seed 3: accuracy = 0.5954


In [ ]:
print("--- Training Improved eLCS ---")
imp_per_seed_acc = []
imp_all_true = []
imp_all_pred = []
imp_last_model = None

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    rng = np.random.RandomState(seed)
    train_subsample_idx = rng.choice(len(X_train), TRAIN_SUBSAMPLE_SIZE, replace=False)
    
    X_train_sub = X_train[train_subsample_idx]
    y_train_sub = y_train[train_subsample_idx]
    X_test_eval = X_test[:TEST_EVAL_SIZE]
    y_test_eval = y_test[:TEST_EVAL_SIZE]

    model = eLCS(
        learning_iterations=LEARNING_ITERATIONS,
        N=POPULATION_SIZE_N,
        do_correct_set_subsumption=True,
        random_state=seed,
    )
    model.fit(X_train_sub, y_train_sub)

    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    
    imp_per_seed_acc.append(acc)
    imp_all_true.extend(y_test_eval.tolist())
    imp_all_pred.extend(y_pred.tolist())
    imp_last_model = model

    print(f"Seed {seed}: accuracy = {acc:.4f}")

## Task 5 - Experiments (Statistical Testing)
Aggregating the overall classification reports for both models and performing McNemar's Test on the pooled paired predictions.

In [ ]:
orig_all_true = np.array(orig_all_true)
orig_all_pred = np.array(orig_all_pred)
imp_all_pred = np.array(imp_all_pred)

print("Classification report (Original eLCS, pooled across all seeds):")
print(classification_report(
    orig_all_true, orig_all_pred, 
    target_names=["Low", "Medium", "High"], zero_division=0
))

print("\nClassification report (Improved eLCS, pooled across all seeds):")
print(classification_report(
    orig_all_true, imp_all_pred, 
    target_names=["Low", "Medium", "High"], zero_division=0
))

# McNemar's Test on pooled predictions
correct_orig = (orig_all_pred == orig_all_true)
correct_imp = (imp_all_pred == orig_all_true)

table = np.zeros((2, 2))
table[0, 0] = np.sum(correct_imp & correct_orig)
table[0, 1] = np.sum(correct_imp & ~correct_orig)
table[1, 0] = np.sum(~correct_imp & correct_orig)
table[1, 1] = np.sum(~correct_imp & ~correct_orig)

mcnemar_res = mcnemar(table, exact=False, correction=True)
print(f"\nMcNemar's test (Improved vs Original): statistic={mcnemar_res.statistic:.3f}, p-value={mcnemar_res.pvalue:.4g}")

## Task 6 - Rule Export and Summary
Exports the final rule population of the last seed and saves the experimental summaries to CSV.

In [ ]:
imp_last_model.export_final_rule_population(
    headerNames=np.array(FEATURE_NAMES),
    className="productivity_class",
    filename="task6_elcs_improved_rules.csv",
    DCAL=True,
)

rule_population = pd.read_csv("task6_elcs_improved_rules.csv")
print(f"Exported {len(rule_population)} rules to task6_elcs_improved_rules.csv")

seed_results = pd.DataFrame({
    "seed": list(range(N_SEEDS)),
    "model_original_acc": orig_per_seed_acc,
    "model_improved_acc": imp_per_seed_acc,
})
seed_results.to_csv("task6_elcs_seed_results.csv", index=False)

summary_row = pd.DataFrame([{
    "model": "Improved eLCS (preprocessed, 15 features)",
    "mean_accuracy": np.mean(imp_per_seed_acc),
    "std_accuracy": np.std(imp_per_seed_acc),
    "n_seeds": N_SEEDS,
    "n_features": X.shape[1],
    "n_rules_last_seed": len(rule_population),
}])
summary_row.to_csv("task6_elcs_improved_summary.csv", index=False)

print("\nSummary saved to task6_elcs_improved_summary.csv:")
print(summary_row)